In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
from scipy import stats

In [4]:
ts = xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/new/ts_ssp5.nc').ts
pr = (xr.open_dataset('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/new/pr_ssp5.nc')).pr*86400*30

In [7]:
ts

<xarray.DataArray 'ts' (model: 47, time: 250, lat: 120, lon: 240)>
[338400000 values with dtype=float32]
Coordinates:
  * lon      (lon) float64 0.0 1.5 3.0 4.5 6.0 ... 352.5 354.0 355.5 357.0 358.5
  * lat      (lat) float64 -89.25 -87.75 -86.25 -84.75 ... 86.25 87.75 89.25
  * time     (time) datetime64[ns] 1850-06-01 1851-06-01 ... 2099-06-01
  * model    (model) object 'CanESM5' 'CanESM5-CanOE' ... 'BCC-CSM2-MR'

In [8]:
weights = np.cos(np.deg2rad(ts.lat))
weights

<xarray.DataArray 'lat' (lat: 120)>
array([0.0130896 , 0.03925982, 0.06540313, 0.09150162, 0.1175374 ,
       0.14349262, 0.1693495 , 0.19509032, 0.22069744, 0.24615329,
       0.27144045, 0.29654157, 0.32143947, 0.34611706, 0.37055744,
       0.39474386, 0.41865974, 0.44228869, 0.46561452, 0.48862124,
       0.51129309, 0.53361452, 0.55557023, 0.57714519, 0.5983246 ,
       0.61909395, 0.639439  , 0.65934582, 0.67880075, 0.69779046,
       0.71630194, 0.73432251, 0.75183981, 0.76884183, 0.78531693,
       0.80125381, 0.81664156, 0.83146961, 0.84572782, 0.85940641,
       0.87249601, 0.88498764, 0.89687274, 0.90814317, 0.91879121,
       0.92880955, 0.93819134, 0.94693013, 0.95501994, 0.96245524,
       0.96923091, 0.97534232, 0.98078528, 0.98555606, 0.98965139,
       0.99306846, 0.99580493, 0.99785892, 0.99922904, 0.99991433,
       0.99991433, 0.99922904, 0.99785892, 0.99580493, 0.99306846,
       0.98965139, 0.98555606, 0.98078528, 0.97534232, 0.96923091,
       0.96245524, 0.95501994, 0.94693013, 0.93819134, 0.92880955,
       0.91879121, 0.90814317, 0.89687274, 0.88498764, 0.87249601,
       0.85940641, 0.84572782, 0.83146961, 0.81664156, 0.80125381,
       0.78531693, 0.76884183, 0.75183981, 0.73432251, 0.71630194,
       0.69779046, 0.67880075, 0.65934582, 0.639439  , 0.61909395,
       0.5983246 , 0.57714519, 0.55557023, 0.53361452, 0.51129309,
       0.48862124, 0.46561452, 0.44228869, 0.41865974, 0.39474386,
       0.37055744, 0.34611706, 0.32143947, 0.29654157, 0.27144045,
       0.24615329, 0.22069744, 0.19509032, 0.1693495 , 0.14349262,
       0.1175374 , 0.09150162, 0.06540313, 0.03925982, 0.0130896 ])
Coordinates:
  * lat      (lat) float64 -89.25 -87.75 -86.25 -84.75 ... 86.25 87.75 89.25
Attributes:
    standard_name:  latitude
    long_name:      latitude
    units:          degrees_north
    axis:           Y

In [9]:
from functions import preproc_funcs as funcs

In [10]:
from dask.diagnostics import ProgressBar

In [11]:
with ProgressBar():
    nino34_model = funcs.detrend_separate_check(ts.sel(lat = slice(-5, 5), lon = slice(-170+360, -120+360)).weighted(weights).mean(('lat', 'lon')).groupby('time.year').mean('time'), dim='year', period=15)

In [12]:
nino34_model

<xarray.DataArray 'ts' (model: 47, year: 250)>
array([[-0.50091011,  0.24715304,  0.40832413, ..., -0.35284261,
        -0.98078553, -0.08190425],
       [-0.50091011,  0.24715304,  0.40832413, ..., -0.35284261,
        -0.98078553, -0.08190425],
       [-0.64615003,  0.89134657, -0.31972806, ...,  0.218147  ,
         1.10387572, -0.9132173 ],
       ...,
       [ 1.39748107, -2.14323852, -1.66676636, ..., -0.83696577,
         1.00356643, -0.90199208],
       [-0.54353218,  0.25750459,  0.1530251 , ..., -0.49173769,
         0.82886037, -1.71845831],
       [-0.86721093,  0.70466954, -0.48898027, ..., -0.18865845,
         0.20868382,  0.21549311]])
Coordinates:
  * model    (model) object 'CanESM5' 'CanESM5-CanOE' ... 'BCC-CSM2-MR'
  * year     (year) int64 1850 1851 1852 1853 1854 ... 2095 2096 2097 2098 2099

In [15]:
pr_det = funcs.detrend_separate_check(pr.groupby('time.year').mean('time'), dim='year', period=15)
pr_det

KeyboardInterrupt: 

In [19]:
with ProgressBar():
    pr_20c_en = pr_det.where(nino34_model > nino34_model.std('year')).sel(time = slice('1900', '2000')).mean('time').load()
    pr_21c_en = pr_det.where(nino34_model > nino34_model.std('year')).sel(time = slice('2000', '2100')).mean('time').load()
    pr_21c_ln = pr_det.where(nino34_model < nino34_model.std('year')).sel(time = slice('1900', '1999')).mean('time').load()
    pr_21c_ln = pr_det.where(nino34_model < nino34_model.std('year')).sel(time = slice('2000', '2100')).mean('time').load()

[                                        ] | 0% Completed | 366.91 us

[                                        ] | 0% Completed | 309.70 ss


KeyboardInterrupt: 

<xarray.Dataset>
Dimensions:  (lon: 240, lat: 120, time: 100, model: 46)
Coordinates:
  * lon      (lon) float64 0.0 1.5 3.0 4.5 6.0 ... 352.5 354.0 355.5 357.0 358.5
  * lat      (lat) float64 -89.25 -87.75 -86.25 -84.75 ... 86.25 87.75 89.25
  * time     (time) datetime64[ns] 1900-06-01 1901-06-01 ... 1999-06-01
  * model    (model) object 'CanESM5' 'CanESM5-CanOE' ... 'BCC-CSM2-MR'
Data variables:
    pr       (model, time, lat, lon) float32 dask.array<chunksize=(46, 100, 120, 240), meta=np.ndarray>